In [ ]:
# 🛠️ 環境セットアップ

# 共通ライブラリのインポート
import sys
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import display
from scipy import signal as sp_signal
import warnings
warnings.filterwarnings('ignore')

# Google Colab環境かどうかを判定
try:
    import google.colab
    IN_COLAB = True
    print("🔧 Google Colab環境で実行中...")
except ImportError:
    IN_COLAB = False
    print("🏠 ローカル環境で実行中")

# ライブラリのセットアップ
if IN_COLAB:
    print("🔧 Google Colab環境を設定中...")
    
    # 必要なパッケージをインストール
    !pip install japanize-matplotlib
    
    # GitHubからライブラリをクローン
    !git clone https://github.com/ggszk/simple-audio-programming.git
    
    # パスを追加
    sys.path.append('/content/simple-audio-programming')

    # ライブラリ設定
    import japanize_matplotlib
    
else:
    print("🔧 ローカル環境を設定中...")

    # パスを追加
    sys.path.append('..')

    # 日本語フォント設定（Mac）
    import platform
    if platform.system() == 'Darwin':
        plt.rcParams['font.family'] = 'Hiragino Sans'
    else:
        plt.rcParams['font.family'] = 'Meiryo'

print("\n🎵 Simple Audio Programming へようこそ！")
print("✅ セットアップ完了")

# 音のプログラミング 第3回: 周波数分析（FFTとスペクトログラム）

前回まで学んだこと：
- 第1回：サイン波の生成、サンプリングの基礎
- 第2回：エンベロープ（ADSR）で音の時間変化をコントロール

**今回の学習目標:**
- フーリエ変換の考え方を理解する
- FFT（高速フーリエ変換）で音の周波数成分を分析する
- スペクトログラムで音の時間変化を可視化する
- 次回のフィルターの準備として「周波数領域」の感覚をつかむ

**所要時間:** 90分

## 🛠️ 今回使用するライブラリ

In [ ]:
from audio_lib import sine_wave, square_wave, sawtooth_wave, adsr, AudioSignal
from audio_lib.notebook import play_sound, plot_waveform, plot_spectrum

print("🛠️ ライブラリを読み込みました")

## 🎯 実習1: サイン波のFFT — 最もシンプルな例

In [ ]:
# 440Hzのサイン波を生成
duration = 1.0
signal_440 = sine_wave(440, duration)

# 時間領域で表示
plot_waveform(signal_440, duration=0.01, title='440Hz サイン波（時間領域）')

# 周波数領域で表示（FFT）
plot_spectrum(signal_440, max_freq=2000, title='440Hz サイン波（周波数領域）')

💡 440Hzにピークが1本だけ立っています。サイン波は単一の周波数成分しか持ちません。

## 🎯 実習2: 和音のFFT — 複数の音を分離する

In [ ]:
# Cメジャーコード（ド・ミ・ソ）を作成
duration = 1.0
c_major_freqs = [261.63, 329.63, 392.00]  # C4, E4, G4
note_names = ['C4 (ド)', 'E4 (ミ)', 'G4 (ソ)']

# 3つのサイン波を重ね合わせ
chord = sine_wave(c_major_freqs[0], duration)
for freq in c_major_freqs[1:]:
    chord = chord + sine_wave(freq, duration)
chord = chord * (1.0 / len(c_major_freqs))  # 音量調整

# 再生
display(play_sound(chord, "Cメジャーコード（ド・ミ・ソ）"))

# 時間領域
plot_waveform(chord, duration=0.02, title='Cメジャーコード（時間領域）')

# 周波数領域
data = chord.data
sr = chord.sample_rate
windowed = data * np.hanning(len(data))
fft_result = np.fft.fft(windowed)
fft_magnitude = np.abs(fft_result)
fft_freq = np.fft.fftfreq(len(data), 1/sr)

positive_idx = fft_freq >= 0
freq_pos = fft_freq[positive_idx]
mag_pos = fft_magnitude[positive_idx]
range_idx = freq_pos <= 1000

plt.figure(figsize=(12, 5))
plt.plot(freq_pos[range_idx], 20*np.log10(mag_pos[range_idx] + 1e-10), 'g-', linewidth=2)
for freq, name in zip(c_major_freqs, note_names):
    plt.axvline(x=freq, color='red', linestyle='--', alpha=0.7, label=f'{name} ({freq:.1f} Hz)')
plt.title('Cメジャーコード（周波数領域）', fontsize=16)
plt.xlabel('周波数 (Hz)', fontsize=12)
plt.ylabel('振幅 (dB)', fontsize=12)
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

💡 時間領域では複雑な波形ですが、周波数領域では3つのピークとしてきれいに分離できます！

## 🎯 実習3: 波形による倍音構造の違い

楽器の音色の違いは**倍音構造**（どの周波数成分がどれだけ含まれているか）で決まります。
異なる波形を FFT で分析して、その違いを見てみましょう。

In [ ]:
# 3種類の波形を比較: サイン波、矩形波、のこぎり波
duration = 1.0
freq = 220  # A3

waves = {
    'サイン波': sine_wave(freq, duration),
    '矩形波': square_wave(freq, duration),
    'のこぎり波': sawtooth_wave(freq, duration),
}

fig, axes = plt.subplots(3, 2, figsize=(14, 10))

for i, (name, sig) in enumerate(waves.items()):
    # 時間領域
    time_samples = int(0.015 * sig.sample_rate)
    t = np.linspace(0, 0.015, time_samples)
    axes[i, 0].plot(t, sig.data[:time_samples], 'b-', linewidth=2)
    axes[i, 0].set_title(f'{name}（時間領域）', fontsize=13)
    axes[i, 0].set_ylabel('振幅')
    axes[i, 0].grid(True, alpha=0.3)
    
    # 周波数領域
    windowed = sig.data * np.hanning(len(sig.data))
    fft_result = np.fft.fft(windowed)
    fft_magnitude = np.abs(fft_result)
    fft_freq = np.fft.fftfreq(len(sig.data), 1/sig.sample_rate)
    pos = fft_freq >= 0
    freq_range = fft_freq[pos] <= 3000
    
    axes[i, 1].plot(fft_freq[pos][freq_range], 
                    20*np.log10(fft_magnitude[pos][freq_range] + 1e-10), 
                    'g-', linewidth=2)
    axes[i, 1].set_title(f'{name}（周波数領域）', fontsize=13)
    axes[i, 1].set_ylabel('振幅 (dB)')
    axes[i, 1].grid(True, alpha=0.3)

axes[2, 0].set_xlabel('時間 (秒)')
axes[2, 1].set_xlabel('周波数 (Hz)')
plt.tight_layout()
plt.show()

In [ ]:
# 音で聴き比べてみよう
for name, sig in waves.items():
    display(play_sound(sig, f"{name} (220Hz)"))

💡 **発見：**
- サイン波 → ピーク1本（倍音なし）
- 矩形波 → 奇数次倍音（220, 660, 1100, ...）が多い
- のこぎり波 → すべての倍音（220, 440, 660, 880, ...）が含まれる

この倍音の違いが「**音色**」の正体です！

## 🎯 実習4: スペクトログラム — 時間と周波数を同時に見る

FFTは「全体の周波数成分」を教えてくれますが、「いつ」その成分があったかはわかりません。

**スペクトログラム**は音を短い区間に分割してFFTを繰り返し適用し、
時間×周波数の2次元マップを作ります。

- 横軸：時間
- 縦軸：周波数
- 色の明るさ：その周波数成分の強さ

In [ ]:
# 音階（ドレミファソラシド）を作成
scale_freqs = [261.63, 293.66, 329.63, 349.23, 392.00, 440.00, 493.88, 523.25]
scale_names = ['C4', 'D4', 'E4', 'F4', 'G4', 'A4', 'B4', 'C5']
note_duration = 0.5

# 各音にエンベロープを適用して連結
scale_data = np.array([])
env = adsr(note_duration, attack=0.05, decay=0.1, sustain=0.8, release=0.15)

for freq in scale_freqs:
    note = sine_wave(freq, note_duration)
    note_with_env = note * env
    scale_data = np.concatenate([scale_data, note_with_env.data])

sr = 44100

# 再生
scale_signal = AudioSignal(scale_data, sr)
display(play_sound(scale_signal, "ドレミファソラシド"))

# スペクトログラムの計算
frequencies, times, Sxx = sp_signal.spectrogram(
    scale_data, fs=sr, window='hann', nperseg=1024, noverlap=512
)

# 可視化
fig, axes = plt.subplots(2, 1, figsize=(14, 8))

# 時間波形
time_axis = np.linspace(0, len(scale_data) / sr, len(scale_data))
axes[0].plot(time_axis, scale_data, 'b-', linewidth=1)
axes[0].set_title('音階（時間波形）', fontsize=16)
axes[0].set_ylabel('振幅')
axes[0].grid(True, alpha=0.3)

# スペクトログラム
freq_limit = 800  # 表示範囲
freq_idx = frequencies <= freq_limit
axes[1].pcolormesh(times, frequencies[freq_idx], 
                   10*np.log10(Sxx[freq_idx] + 1e-10), 
                   shading='gouraud', cmap='viridis')
axes[1].set_title('スペクトログラム（時間-周波数分析）', fontsize=16)
axes[1].set_xlabel('時間 (秒)')
axes[1].set_ylabel('周波数 (Hz)')

# 理論値を水平線で表示
for freq, name in zip(scale_freqs, scale_names):
    axes[1].axhline(y=freq, color='white', linestyle='--', alpha=0.5, linewidth=0.8)

plt.tight_layout()
plt.show()

各音の周波数が階段状に上がっていく様子が見えます。明るい色はその時刻にその周波数の音が鳴っていることを示しています。

## 🎯 実習5: 周波数分析の活用 — フィルターの予告

次回のレッスンでは**フィルター**を学びます。
フィルターは「特定の周波数成分を削る（または強調する）」処理です。

ここでは簡単な例として、のこぎり波の高い周波数成分を手動で削ってみましょう。

In [ ]:
# のこぎり波（倍音が豊富）
saw = sawtooth_wave(220, 1.0)

# FFTで周波数領域に変換
fft_data = np.fft.fft(saw.data)
fft_freq = np.fft.fftfreq(len(saw.data), 1/saw.sample_rate)

# 1000Hz以上の成分をゼロにする（簡易ローパスフィルター）
cutoff = 1000
fft_filtered = fft_data.copy()
fft_filtered[np.abs(fft_freq) > cutoff] = 0

# 周波数領域から時間領域に戻す（逆FFT）
filtered_data = np.real(np.fft.ifft(fft_filtered))
filtered_signal = AudioSignal(filtered_data, saw.sample_rate)

# 比較
display(play_sound(saw, "のこぎり波（フィルターなし）"))
display(play_sound(filtered_signal, f"のこぎり波（{cutoff}Hz以上カット）"))

# スペクトラム比較
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

for ax, sig, title in [(axes[0], saw, 'フィルター前'), 
                        (axes[1], filtered_signal, f'フィルター後（{cutoff}Hz以上カット）')]:
    windowed = sig.data * np.hanning(len(sig.data))
    fft_r = np.fft.fft(windowed)
    fft_m = np.abs(fft_r)
    fft_f = np.fft.fftfreq(len(sig.data), 1/sig.sample_rate)
    pos = fft_f >= 0
    rng = fft_f[pos] <= 3000
    ax.plot(fft_f[pos][rng], 20*np.log10(fft_m[pos][rng] + 1e-10), 'g-', linewidth=2)
    ax.axvline(x=cutoff, color='red', linestyle='--', alpha=0.7, label=f'カットオフ {cutoff}Hz')
    ax.set_title(title, fontsize=13)
    ax.set_xlabel('周波数 (Hz)')
    ax.set_ylabel('振幅 (dB)')
    ax.legend()
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

💡 高い周波数成分をカットすると、音が柔らかくなります。これがフィルターの基本的な動作です。次回のレッスンで、もっと高度なフィルターを学びます。

## 🏆 チャレンジ課題

In [ ]:
# チャレンジ1: 好きな和音を作ってFFTで分析しよう
# ヒント: Am（ラ・ド・ミ）= [440, 523.25, 659.26]
#         G （ソ・シ・レ）= [392.00, 493.88, 587.33]

my_freqs = [440, 523.25, 659.26]  # ここを変更してください
my_chord = sine_wave(my_freqs[0], 1.0)
for f in my_freqs[1:]:
    my_chord = my_chord + sine_wave(f, 1.0)
my_chord = my_chord * (1.0 / len(my_freqs))

display(play_sound(my_chord, "マイ和音"))
plot_spectrum(my_chord, max_freq=1500, title='マイ和音の周波数分析')

In [ ]:
# チャレンジ2: カットオフ周波数を変えて音の変化を体験しよう
# のこぎり波の倍音をどこまで削ると音色が変わる？

saw = sawtooth_wave(220, 1.5)

for cutoff in [500, 1000, 2000, 4000]:
    fft_data = np.fft.fft(saw.data)
    fft_freq = np.fft.fftfreq(len(saw.data), 1/saw.sample_rate)
    fft_filtered = fft_data.copy()
    fft_filtered[np.abs(fft_freq) > cutoff] = 0
    filtered = AudioSignal(np.real(np.fft.ifft(fft_filtered)), saw.sample_rate)
    display(play_sound(filtered, f"カットオフ {cutoff}Hz"))

## 📚 今日のまとめ

### 学んだこと
1. **フーリエ変換**: 時間領域の信号を周波数領域に変換する技法
2. **FFT**: `np.fft.fft()` による高速な周波数分析
3. **倍音構造**: 波形の種類による周波数成分の違いが「音色」の正体
4. **スペクトログラム**: 時間×周波数の2次元マップで音の変化を可視化
5. **周波数フィルタリング**: 特定の周波数成分を削ることで音色を変える

### 使った技術
- `np.fft.fft()` / `np.fft.ifft()`: フーリエ変換・逆変換
- `np.fft.fftfreq()`: 周波数軸の計算
- `scipy.signal.spectrogram()`: スペクトログラムの計算
- 窓関数（ハニング窓）: スペクトル漏れの防止

### 次回予告
次回は「**フィルターと音色デザイン**」を学びます。
今回学んだ周波数分析の知識を活かして、
ローパスフィルターで音色をコントロールし、楽器らしい音を作ります！

---
**お疲れさまでした！** 🎉